# Fashion Trend Forecasting: Predictive Analytics

**Goal:** Demonstrate a basic time-series forecasting workflow for a fashion-interest signal.

**Important:** The included data is synthetic and only makes the notebook runnable. It is not real Google Trends data, consumer behaviour or retailer sales. Replace it with a documented, permitted public series before reporting market findings.

## Workflow
1. Load and validate monthly data.
2. Visualise trend and seasonality.
3. Create a chronological train/test split.
4. Compare a seasonal-naive baseline.
5. Evaluate MAE and RMSE.
6. Explain limitations and next steps.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_PATH = Path("../data/fashion_search_interest_demo.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["month"]).sort_values("month")
target = "search_interest_index"

assert df["month"].is_monotonic_increasing
assert not df["month"].duplicated().any()
assert df[target].notna().all()

print("Rows:", len(df))
print("Date range:", df["month"].min().date(), "to", df["month"].max().date())
print("Data status:", df["data_status"].unique())
display(df.head())
display(df[target].describe())

## 1. Explore the time series
Look for trend, recurring seasonal behaviour, sudden spikes and missing periods.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["month"], df[target], marker="o", markersize=3)
plt.title("Monthly Search-Interest Index (Synthetic Demo)")
plt.xlabel("Month")
plt.ylabel("Illustrative relative index")
plt.tight_layout()
plt.show()

## 2. Chronological train/test split
Time series should not be randomly shuffled. Here, the final 12 months are held out for testing.

In [ ]:
if len(df) < 24:
    raise ValueError("At least 24 monthly observations are recommended for this seasonal demo.")

train = df.iloc[:-12].copy()
test = df.iloc[-12:].copy()

# Seasonal-naive: predict each test month using the value from 12 months earlier.
pred = df[target].iloc[-24:-12].to_numpy()

mae = mean_absolute_error(test[target], pred)
rmse = mean_squared_error(test[target], pred) ** 0.5
print(f"Seasonal-naive MAE: {mae:.2f}")
print(f"Seasonal-naive RMSE: {rmse:.2f}")

## 3. Plot actual versus forecast

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train["month"], train[target], label="Training data")
plt.plot(test["month"], test[target], label="Actual test values")
plt.plot(test["month"], pred, marker="o", linestyle="--", label="Seasonal-naive forecast")
plt.title("Actual vs Seasonal-Naive Forecast (Synthetic Demo)")
plt.xlabel("Month")
plt.ylabel("Illustrative relative index")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Interpretation and limitations

- MAE reports average absolute forecast error in the target's units.
- RMSE penalises large errors more heavily.
- These metrics are meaningful only relative to a baseline and the target scale.
- The data is synthetic, so the scores do not represent real fashion-market forecasting performance.
- For a real project, retrieve a consistent public time series, document query settings and geography, and evaluate with rolling-origin validation.
- Compare the baseline with ETS or SARIMA only if enough history and stable seasonal patterns exist.
- Search interest is a proxy for attention, not a direct measure of purchases or sales.